# 01 — Dataset exploration and data integrity

**Purpose:** Inspect the downloaded competition dataset before modeling.

This notebook inventories images and CSVs, displays representative photographs, and explores grain-size labels if their columns match the expected schema.

**Important:** Image folders may contain multiple photographs per soil sample. Do not split individual photographs randomly into training and validation.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from src.metric import TARGET_COLUMNS, DIAMETERS

DATA_DIR = ROOT / "data"
print("Project:", ROOT.resolve())
print("Data directory:", DATA_DIR.resolve())
if not DATA_DIR.exists():
    print("Download the official Kaggle dataset into data/ before proceeding.")

## 1. Inventory data files

Review folder names and the raw schemas first. We do not assume camera metadata matches image filenames until checked.

In [ ]:
if DATA_DIR.exists():
    files = sorted(p for p in DATA_DIR.rglob("*") if p.is_file())
    print("Total files:", len(files))
    print("By extension:\n", pd.Series([p.suffix.lower() for p in files]).value_counts().to_string() if files else "No files")
    display(pd.DataFrame({"file": [str(p.relative_to(DATA_DIR)) for p in files[:40]],
                          "bytes": [p.stat().st_size for p in files[:40]]}))
else:
    files = []

In [ ]:
csv_files = [p for p in files if p.suffix.lower() == ".csv"]
for path in csv_files:
    print("\nCSV:", path.relative_to(DATA_DIR))
    try:
        frame = pd.read_csv(path)
        print("shape:", frame.shape)
        display(frame.head(3))
        print("columns:", frame.columns.tolist())
    except Exception as exc:
        print("Could not parse:", exc)

## 2. Preview a few photographs

These are previews for exploration only. Do not infer image scale from display size.

In [ ]:
image_files = [p for p in files if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}]
if image_files:
    selected = image_files[:min(6, len(image_files))]
    fig, axes = plt.subplots(1, len(selected), figsize=(3 * len(selected), 3))
    axes = np.atleast_1d(axes)
    for ax, path in zip(axes, selected):
        try:
            with Image.open(path) as im:
                ax.imshow(im.convert("RGB"))
                ax.set_title(f"{path.name[:25]}\n{im.size[0]}x{im.size[1]}", fontsize=8)
        except Exception as exc:
            ax.text(.5,.5,str(exc)[:50], ha="center")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No images found.")

## 3. Inspect target distributions

The schema below is provisional. Check the printed columns against the official data before adapting the loader.

In [ ]:
labels_candidates = [p for p in csv_files if "label" in p.name.lower() or "train" in p.name.lower()]
if labels_candidates:
    label_path = labels_candidates[0]
    labels = pd.read_csv(label_path)
    print("Candidate:", label_path.name, "shape:", labels.shape)
    if set(TARGET_COLUMNS).issubset(labels.columns):
        curves = labels[TARGET_COLUMNS].apply(pd.to_numeric, errors="coerce")
        print("Missing target entries:", int(curves.isna().sum().sum()))
        print("Unique sample IDs:", labels["sample_id"].nunique() if "sample_id" in labels else "unknown")
        fig, ax = plt.subplots(figsize=(9, 5))
        for curve in curves.to_numpy():
            ax.plot(DIAMETERS, curve, alpha=0.45)
        ax.set_xscale("log")
        ax.set(xlabel="Grain diameter (mm)", ylabel="Cumulative passing mass (%)", ylim=(0, 100))
        ax.grid(alpha=0.25)
        plt.show()
    else:
        print("Target columns are not an exact match; inspect actual CSV fields above.")
else:
    print("No candidate labels CSV found.")

## Audit checklist

- [ ] Confirm independent soil sample count and sample IDs.
- [ ] Verify photo-to-sample mappings for *all* image files.
- [ ] Inspect magnification, focus, lighting, and duplicate photos.
- [ ] Inspect PPM/camera metadata and join keys.
- [ ] Inspect label monotonicity, missing data, and ranges.
- [ ] Ensure no held-out sample photos can leak into training.

**Next notebook:** `02_baseline_and_validation.ipynb`.